In [285]:
from collections import Counter, defaultdict
import pandas as pd
from scipy.sparse import coo_matrix
import numpy as np
import matplotlib.pyplot as plt

import Levenshtein

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

In [91]:
# training data sources
entities = pd.read_csv('data/archive/train/train_source1.tsv', sep = '\t').set_index('entity_id').sort_index()
entities = entities.fillna("")
s2 = pd.read_csv('data/archive/train/train_source2.tsv', sep = '\t')
s3 = pd.read_csv('data/archive/train/train_source3.tsv', sep = '\t')

s2['source'] = 'source1'
s3['source'] = 'source2'
datasets = pd.concat((s2, s3), axis = 0).set_index('entity_id').sort_index()
datasets = datasets.fillna("")
del s2, s3

# ground truth
gt = pd.read_csv('data/archive/train/train_ground_truth.tsv', sep = '\t')
gt = gt.fillna("")

#### Create a sparse data base for the ground truths

In [20]:
row_index = {}
col_index = {}
graph_dat = defaultdict(lambda: Counter())

for row in gt.values:
    
    entity_id = row[0]
    if entity_id not in row_index:
        row_index[entity_id] = len(row_index)
    ridx = row_index[entity_id]

    if row[1] != '':
        for matched_entity_id in row[1].split(','):
            if matched_entity_id not in col_index:
                col_index[matched_entity_id] = len(col_index)
            colidx = col_index[matched_entity_id]
            graph_dat[ridx][colidx] += 1
row = []
col = []
data = []

for k1 in graph_dat:
    for k2 in graph_dat[k1]:
        row.append(k1)
        col.append(k2)
        data.append(graph_dat[k1][k2])

sps = coo_matrix((data, (row, col)), shape=(len(row_index), len(col_index)))

In [21]:
match_count = sps.sum(axis=1)
match_count = np.array(match_count).reshape(-1)
matches = Counter()
total = 0
for el in match_count:
    matches[el] += 1
    total += 1

In [22]:
for el in range(11):
    print(f'{el:.<10}...{matches[el] / total:.>10.1%}')

0..................5.6%
1..................5.4%
2.................17.0%
3.................24.1%
4.................21.9%
5.................14.6%
6..................7.5%
7..................2.9%
8..................0.8%
9..................0.2%
10.................0.0%


#### Create reverse lookup to match back to entity id

In [23]:
reverse_lookup = {}
row_index_rev = {v: k for k, v in row_index.items()}
col_index_rev = {v: k for k, v in col_index.items()}

for k1 in graph_dat:
    for k2 in graph_dat[k1]:
        reverse_lookup[col_index_rev[k2]] = row_index_rev[k1]

In [7]:
print("Every matched entity in the data table only matches to one source entity -- reverse look up is possible")
print(f" - reverse lookup size: {len(reverse_lookup):,d}")
print(f" - sum of Edges in Sparse Dataset: {sps.astype('bool').astype('int').sum():,d}")

Every matched entity in the data table only matches to one source entity -- reverse look up is possible
 - reverse lookup size: 7,638,365
 - sum of Edges in Sparse Dataset: 7,638,365


In [25]:
# data clean up
del graph_dat
del sps
del row_index
del col_index
del row_index_rev
del col_index_rev

In [9]:
matches

Counter({np.int64(3): 530841,
         np.int64(4): 484115,
         np.int64(2): 375212,
         np.int64(5): 321957,
         np.int64(6): 164868,
         np.int64(0): 123247,
         np.int64(1): 119157,
         np.int64(7): 63968,
         np.int64(8): 18680,
         np.int64(9): 4205,
         np.int64(10): 534,
         np.int64(11): 37})

#### Data Exploration

In [10]:
n_entities = entities.shape[0]
n_s1 = datasets[datasets['source'] == 'source1'].shape[0]
n_s2 = datasets[datasets['source'] == 'source2'].shape[0]

print("Total Row Count Information")
print(f" - {n_entities:,d} source businesses to be matched")
print(f" - {n_s1:,d} businesses to match in dataset 1")
print(f" - {n_s2:,d} businesses to match in dataset 2")
print("Ground Truth Information")

print()
print("Total Potential Training Volume")
print(f" - {n_entities * n_s1:,d} business combinations in dataset 1")
print(f" - {n_entities * n_s2:,d} business combinations in dataset 2")
print(f" - {n_entities * n_s1 + n_entities * n_s2:,d} business combinations")

Total Row Count Information
 - 2,206,821 source businesses to be matched
 - 5,034,616 businesses to match in dataset 1
 - 5,285,603 businesses to match in dataset 2
Ground Truth Information

Total Potential Training Volume
 - 11,110,496,315,736 business combinations in dataset 1
 - 11,664,379,698,063 business combinations in dataset 2
 - 22,774,876,013,799 business combinations


#### Data Processing

#### Featurization
Because we are going to learn on the fly we need a featurization function. This will take two entity ids and create features to estimate if they are matches or not.

In [11]:
entity_id = 'S1-100000221'
match_id = 'S2-100000037'

In [12]:
datasets.loc[match_id]

business_name                     PREMIER INC  PRODUCTS LEARNING
business_address    VA, SPOTSYLVANIA, 11604-11606 PINE HOLLOW LN
country                                                       US
source                                                   source1
Name: S2-100000037, dtype: str

In [263]:
def _basic_string_cleaning(string):
    return string.strip().lower()

def _clean_business_name(name):
    return _basic_string_cleaning(name)

def _clean_address(address):
    return _basic_string_cleaning(address)

def _length(string, cleaning = None):
    if cleaning is not None:
        string = cleaning(string)
    return len(string)

def _words(string, cleaning = None):
    if cleaning is not None:
        string = cleaning(string)
    return len(set(string))

def _same_check(v1, v2, cleaning = None):
    if cleaning is not None:
        v1 = cleaning(v1)
        v2 = cleaning(v2)
    return float(v1 == v2)

def _distance(v1, v2, cleaning = None):
    if cleaning is not None:
        v1 = cleaning(v1)
        v2 = cleaning(v2)
    return Levenshtein.distance(v1, v2)

def _word_match(v1, v2, cleaning = None):
    if cleaning is not None:
        v1 = cleaning(v1)
        v2 = cleaning(v2)
    v1 = set(v1.split(' '))
    v2 = set(v2.split(' '))
    return len(v1.intersection(v2))
        

def featurize(check):
    entity, match = check 
    r1 = entities.loc[entity]
    r2 = datasets.loc[match]

    feature = []

    # truth check on fields
    feature.append(_same_check(r1['business_name'], r2['business_name'], _basic_string_cleaning))
    feature.append(_same_check(r1['business_address'], r2['business_address'], _basic_string_cleaning))

    # check if string lengths are close. Note this is more valuable as an interactive feature
    feature.append(_length(r1['business_name'], _basic_string_cleaning))
    feature.append(_length(r2['business_name'], _basic_string_cleaning))
    feature.append(_length(r1['business_address'], _basic_string_cleaning))
    feature.append(_length(r2['business_address'], _basic_string_cleaning))

    # check if word lengths are close. Note this is more valuable as an interactive feature
    feature.append(_words(r1['business_name'], _basic_string_cleaning))
    feature.append(_words(r2['business_name'], _basic_string_cleaning))
    feature.append(_words(r1['business_address'], _basic_string_cleaning))
    feature.append(_words(r2['business_address'], _basic_string_cleaning))

    # check txt distance
    feature.append(_word_match(r1['business_name'], r2['business_name'], _basic_string_cleaning))
    feature.append(_word_match(r1['business_address'], r2['business_address'], _basic_string_cleaning))

    # check word length match
    feature.append(_word_match(r1['business_name'], r2['business_name'], _basic_string_cleaning))
    feature.append(_word_match(r1['business_address'], r2['business_address'], _basic_string_cleaning))
    
    return feature

#### Model and Training
There are total 22 trillion possible combinations but only 7 million of these are viable edges. The best way to train this data set is to include all positive training cases and incorporate negative training cases randomly. I am going to choose an "online" gradient descent method to learn negative features because this will allow us to start to understand how many negative examples to train on. 

- Positive cases can be loaded in memory for training
- Negative cases have to be selected randomly from list, checked to make sure not valid match and then built on the fly

In [299]:
positive_cases = [(reverse_lookup[k], k) for k in reverse_lookup]
entity_list = np.array(entities.index)
match_list = np.array([k for k in reverse_lookup])
rng = np.random.default_rng()

def get_negative_case():
    e, m = '', ''
    while e == reverse_lookup.get(m, ''):
        e = rng.choice(entity_list)
        m = rng.choice(match_list)
    return e, m

In [300]:
X_pos = pd.DataFrame({'inputs': positive_cases[:100000]})
X_pos['features'] = X_pos['inputs'].apply(featurize)
X_pos['target'] = 1

X_neg = pd.DataFrame({'inputs': [get_negative_case() for _ in range(100000)]})
X_neg['features'] = X_neg['inputs'].apply(featurize)
X_neg['target'] = 0

X = pd.concat((X_pos, X_neg), axis = 0)
X = X.sample(frac=1).reset_index(drop=True)

In [301]:
X['features'].head(3)

0    [0.0, 0.0, 25, 21, 95, 97, 14, 13, 29, 29, 1, ...
1    [0.0, 0.0, 32, 24, 59, 31, 16, 14, 21, 18, 3, ...
2    [1.0, 0.0, 29, 29, 41, 48, 15, 15, 21, 23, 4, ...
Name: features, dtype: object

In [302]:
X['features'][0]

[0.0, 0.0, 25, 21, 95, 97, 14, 13, 29, 29, 1, 12, 1, 12]

In [303]:
X['target'].head(3)

0    1
1    1
2    1
Name: target, dtype: int64

#### Training

In [304]:
model = RandomForestClassifier()

X_train, X_test = train_test_split(X, test_size = .2)
model.fit(X_train['features'].tolist(), X_train['target'])
p_train = model.predict(X_train['features'].tolist())
p_test = model.predict(X_test['features'].tolist())

#### Evaluation

In [305]:
precision_train = precision_score(X_train['target'], p_train)
recall_train = recall_score(X_train['target'], p_train)
f1_train = f1_score(X_train['target'], p_train)

precision_test = precision_score(X_test['target'], p_test)
recall_test = recall_score(X_test['target'], p_test)
f1_test = f1_score(X_test['target'], p_test)

print("Training Error")
print(f"{'Precision Score:': <25} {precision_train:>10,.1%}")
print(f"{'Recall Score:': <25} {recall_train:>10,.1%}")
print(f"{'F1 Score:': <25} {f1_train:>10.1%}")
      
print("Testing Error")
print(f"{'Precision Score:': <25} {precision_test:>10,.1%}")
print(f"{'Recall Score:': <25} {recall_test:>10,.1%}")
print(f"{'F1 Score:': <25} {f1_test:>10.1%}")

Training Error
Precision Score:              100.0%
Recall Score:                 100.0%
F1 Score:                     100.0%
Testing Error
Precision Score:               99.5%
Recall Score:                  99.4%
F1 Score:                      99.4%


##### False Positive Examination

In [306]:
mask = (p_test == 1) & (X_test['target'] == 0)
for a, b in X_test[mask]['inputs'][:3]:

    print(f'Comparing {a} to {b}')
    print('---------------------')
    print(entities.loc[a])
    print('---------------------')
    print(datasets.loc[b])
    print('')
    print('')
    print('')

Comparing S1-542432067 to S2-104343116
---------------------
business_name                                    Leon Ethereum Inc
business_address    OH, 13200 Columbiana Canfield Road, Columbiana
country                                                         US
Name: S1-542432067, dtype: str
---------------------
business_name        Secure Secure Molecular Inc
business_address    322 AVERY ROAD, GARRISON, NY
country                                       US
source                                   source1
Name: S2-104343116, dtype: str



Comparing S1-492311168 to S2-814432006
---------------------
business_name                           Long & Lilly Textiles LLC
business_address    2370 Alta Garden Lane, Unit A, Sacramento, CA
country                                                        US
Name: S1-492311168, dtype: str
---------------------
business_name       Date, Zhao & Horner Perfect
business_address                               
country                                      US

##### False Negative Examination

In [307]:
mask = (p_test == 0) & (X_test['target'] == 1)
for a, b in X_test[mask]['inputs'][:3]:

    print(f'Comparing {a} to {b}')
    print('---------------------')
    print(entities.loc[a])
    print('---------------------')
    print(datasets.loc[b])
    print('')
    print('')
    print('')

Comparing S1-913078709 to S3-116254950
---------------------
business_name           Prime Aerospace Enterprises
business_address    11-12 143 Place, Whitestone, NY
country                                          US
Name: S1-913078709, dtype: str
---------------------
business_name                        Prime-Aerospace Enterprises
business_address    143 Pl, null, Whitestonee Township, New York
country                                                       US
source                                                   source2
Name: S3-116254950, dtype: str



Comparing S1-30366118 to S3-699164739
---------------------
business_name                 S V & V Hartford Inc.
business_address    333 Parks Circle, Harrogate, TN
country                                          US
Name: S1-30366118, dtype: str
---------------------
business_name                       Rizanex | www.rizanex.com
business_address    Tennessee, Harrogate CTIY, 0333 Parks Cir
country                                     